# Two-Tower Recommender — Train, Log & Deploy

End-to-end pipeline in a single notebook: Feature View training sets → PyTorch two-tower model → MLflow logging → item embeddings to Delta / Vector Search → model serving endpoint with VS retrieval.

In [0]:
%pip install databricks-feature-engineering>=0.16.0 databricks-vectorsearch torch scipy --quiet
dbutils.library.restartPython()

In [0]:
import yaml
import os
import sys
import mlflow
import numpy as np
import pandas as pd
import math
import torch
import torch.nn as nn
import torch.nn.functional as TF
from torch.utils.data import Dataset, DataLoader
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from databricks.feature_engineering import FeatureEngineeringClient

In [0]:
import os
import yaml
import mlflow
from databricks.feature_engineering import FeatureEngineeringClient

from pyspark.sql import functions as F
from pyspark.sql.window import Window

fe = FeatureEngineeringClient()

# Load project config
config_path = os.path.join(os.getcwd(), "config.yml")
with open(config_path) as f:
    config = yaml.safe_load(f)

CATALOG = config["catalog"]
SCHEMA = config["schema"]
TABLE = f"{CATALOG}.{SCHEMA}.{config['ecommerce_table']}"
PRODUCT_TABLE = f"{CATALOG}.{SCHEMA}.{config['product_table']}"
PRODUCT_TRAIN_TABLE = f"{CATALOG}.{SCHEMA}.{config['product_train_table']}"
SESSION_TRAIN_TABLE = f"{CATALOG}.{SCHEMA}.{config['session_train_table']}"
EVALUATION_TABLE = f"{CATALOG}.{SCHEMA}.{config['eval_table']}"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{config['volume']}"
EMBED_DIM = config["embed_dim"]
BATCH_SIZE = config["batch_size"]
NUM_EPOCHS = config["num_epochs"]

VS_ENDPOINT = config["vs_endpoint"]
VS_INDEX = f"{CATALOG}.{SCHEMA}.{config['vs_index_name']}"
EMBEDDING_TABLE = f"{CATALOG}.{SCHEMA}._two_tower_item_embeddings"

mlflow.set_experiment(config["mlflow_experiment"])
print(f"Catalog: {CATALOG}")
print(f"Schema:  {SCHEMA}")
print(f"Table:   {TABLE}")
print("Setup complete")

In [0]:
# ---------------------------------------------------------------------------
# Retrieve all 25 registered features from UC (18 Lakebase + 7 RequestSource).
# Split into item (15) and session (10) groups for per-tower fe.log_model().
# ---------------------------------------------------------------------------

# Item tower features: product-level Lakebase + product_id RequestSource
ITEM_FEATURE_NAMES = [
    "item_price", "item_category_l1", "item_category_l2",
    "item_category_l3", "item_brand", "item_log_price",
    "item_view_count", "item_cart_count", "item_purchase_count",
    "item_recency", "category_l1_min_price", "category_l1_max_price",
    "category_l2_min_price", "category_l2_max_price",
    "product_id",
]

# Session/user tower features: session-level Lakebase + session RequestSource
SESSION_FEATURE_NAMES = [
    "session_event_count", "session_avg_price",
    "session_price_stddev", "session_total_value",
    "category_l1", "category_l2",
    "relevance", "session_cart_value", "hour_of_day", "day_of_week",
]

item_features = [fe.get_feature(full_name=f"{CATALOG}.{SCHEMA}.{n}") for n in ITEM_FEATURE_NAMES]
session_features = [fe.get_feature(full_name=f"{CATALOG}.{SCHEMA}.{n}") for n in SESSION_FEATURE_NAMES]
all_features = item_features + session_features
print(f"Retrieved {len(all_features)} features ({len(item_features)} item + {len(session_features)} session)")

#### Why point-in-time joins matter here

`fe.create_training_set` joins features onto labeled events using **point-in-time correctness** — each row only sees feature values computed from data that existed *before* that event's timestamp. This is critical for two reasons:

1. **Rolling aggregates leak future signal.** Product-level features like `item_view_count` and `item_purchase_count` are 7-day SlidingWindow aggregates. A naive join would attach counts that include events *after* the training row, telling the model which items become popular — information unavailable at serving time.

2. **Session features must reflect partial state.** Features like `session_event_count` and `session_avg_price` describe the user's session *up to the current event*. Without point-in-time enforcement, a mid-session event could see end-of-session totals, leaking the user's future intent.

Without this, the model trains on features it will never have access to in production, inflating offline metrics while degrading real-world performance.

In [0]:
full_df = spark.table(TABLE)

w_session = Window.partitionBy("user_session")
w_prior = (
    Window.partitionBy("user_session")
    .orderBy("event_time")
    .rowsBetween(Window.unboundedPreceding, -1)
)

labeled_df = (
    full_df
    .filter(F.to_date("event_time").between(config["training_start_day"], config["evalauation_end_day"]))
    .withColumn("category_l1", F.coalesce(F.split("category_code", "\\.").getItem(0), F.lit("unknown")))
    .withColumn("category_l2", F.coalesce(F.split("category_code", "\\.").getItem(1), F.lit("unknown")))
    .select(
        "product_id", "category_l1", "category_l2",
        "user_session", "event_time", "event_type", "price",
    )
    .withColumn("session_first_date", F.min(F.to_date("event_time")).over(w_session))
    .withColumn("relevance",
        F.when(F.col("event_type") == "purchase", 3)
         .when(F.col("event_type") == "cart", 2)
         .otherwise(1))
    .withColumn("session_cart_value", F.coalesce(
        F.sum(
            F.when(F.col("event_type").isin("cart", "purchase"), F.col("price"))
             .otherwise(0)
        ).over(w_prior),
        F.lit(0.0)))
    .withColumn("hour_of_day", F.hour("event_time"))
    .withColumn("day_of_week", F.dayofweek("event_time") - 1)
)

# Train/eval split from config (using original 2019 dates for filtering)
labeled_train = labeled_df.filter(F.col("session_first_date") <= config["training_end_day"])
labeled_eval = labeled_df.filter(F.col("session_first_date") >= config["evaluation_start_day"])

# Shift event_time forward to align with Feature View source tables.
# The product + aggregates tables were created with DELTA_DAYS=2491 shift
# (notebook 01), so their event_time is in 2026. The raw clickstream has
# original 2019 dates. Without this shift, point-in-time Feature View
# joins return NULL for all product features.
DELTA_DAYS = 2491
labeled_train = labeled_train.withColumn("event_time", F.expr(f"event_time + INTERVAL {DELTA_DAYS} DAYS"))
labeled_eval = labeled_eval.withColumn("event_time", F.expr(f"event_time + INTERVAL {DELTA_DAYS} DAYS"))

# --- Per-tower training sets (objects for fe.log_model feature lineage) ---
item_training_set = fe.create_training_set(
    df=labeled_train, features=item_features,
    label="event_type", exclude_columns=["session_first_date", "price"],
)
session_training_set = fe.create_training_set(
    df=labeled_train, features=session_features,
    label="event_type", exclude_columns=["session_first_date", "price"],
)

# --- Combined training set (all features, for training loop) ---
training_set = fe.create_training_set(
    df=labeled_train, features=all_features,
    label="event_type", exclude_columns=["session_first_date", "price"],
)
training_df = training_set.load_df()

# Persist split tables by column selection from the combined DataFrame
_SESSION_FV_COLS = {"session_event_count", "session_avg_price", "session_price_stddev", "session_total_value"}
_ITEM_FV_COLS = {
    "item_price", "item_category_l1", "item_category_l2", "item_category_l3",
    "item_brand", "item_log_price", "item_view_count", "item_cart_count",
    "item_purchase_count", "item_recency", "category_l1_min_price",
    "category_l1_max_price", "category_l2_min_price", "category_l2_max_price",
}
product_cols = [c for c in training_df.columns if c not in _SESSION_FV_COLS]
session_cols = [c for c in training_df.columns if c not in _ITEM_FV_COLS]

training_df.select(product_cols).write.mode("overwrite").saveAsTable(PRODUCT_TRAIN_TABLE)
training_df.select(session_cols).write.mode("overwrite").saveAsTable(SESSION_TRAIN_TABLE)

# --- Eval set (all features) ---
eval_set = fe.create_training_set(
    df=labeled_eval, features=all_features,
    label="event_type", exclude_columns=["session_first_date", "price"],
)
eval_df = eval_set.load_df()
eval_df.write.mode("overwrite").saveAsTable(EVALUATION_TABLE)

# For downstream cells
train_df = training_df
eval_df = spark.table(EVALUATION_TABLE)

product_count = spark.table(PRODUCT_TRAIN_TABLE).count()
session_count = spark.table(SESSION_TRAIN_TABLE).count()
eval_count = spark.table(EVALUATION_TABLE).count()
print(f"Product train: {product_count:,} rows \u2192 {PRODUCT_TRAIN_TABLE}")
print(f"Session train: {session_count:,} rows \u2192 {SESSION_TRAIN_TABLE}")
print(f"Eval set:      {eval_count:,} rows \u2192 {EVALUATION_TABLE}")

#### Feature transformation strategy

The raw Feature View columns from Lakebase are not model-ready — they require derived numerics, vocab lookups, and cyclical encodings before the two-tower model can consume them. These transformations live in a single `transforms.py` module that serves as the **single source of truth** for both training and serving:

* **Training**: imported here and applied distributedly via `mapInPandas`, which passes partitions of the Spark DataFrame through the pandas transform functions. Vocabs are captured in a closure and serialized via cloudpickle.
* **Serving**: the same `transforms.py` is packaged as an MLflow `code_paths` artifact. The pyfunc's `predict()` method imports and calls the same functions on the single-row DataFrame it receives from the FE serving wrapper.

This guarantees identical feature engineering in both paths. 

In [0]:
# ---------------------------------------------------------------------------
# Import shared transforms from transforms.py (single source of truth for
# training via mapInPandas and serving via pyfunc code_paths artifact).
# ---------------------------------------------------------------------------

# Derive project root from config_path (already resolved in cell 3)
_project_root = os.path.normpath(os.path.dirname(config_path))
sys.path.insert(0, _project_root)
from transforms import transform_item_features, transform_user_features, transform_all_features

# Path to transforms.py for MLflow code_paths (packaged into serving container)
transforms_module_path = os.path.join(_project_root, "transforms.py")
print(f"transforms.py: {transforms_module_path}")

# --- Build vocabulary maps from TRAIN only (0 = <UNK>) ---
def build_vocab(sdf, col):
    vals = [row[col] for row in sdf.select(col).distinct().collect() if row[col] is not None]
    return {v: i + 1 for i, v in enumerate(sorted(vals))}

item_vocab = build_vocab(train_df, "product_id")
category_vocab = build_vocab(train_df, "item_category_l1")
category_l2_vocab = build_vocab(train_df, "item_category_l2")
category_l3_vocab = build_vocab(train_df, "item_category_l3")
brand_vocab = build_vocab(train_df, "item_brand")

vocabs = {
    "item_vocab": item_vocab,
    "category_vocab": category_vocab,
    "category_l2_vocab": category_l2_vocab,
    "category_l3_vocab": category_l3_vocab,
    "brand_vocab": brand_vocab,
}
print(f"Vocabs — items: {len(item_vocab)}, cat_l1: {len(category_vocab)}, "
      f"cat_l2: {len(category_l2_vocab)}, cat_l3: {len(category_l3_vocab)}, "
      f"brands: {len(brand_vocab)}")

# --- Apply transforms via mapInPandas (distributed, scales to millions) ---
from pyspark.sql.types import StructType, StructField, DoubleType, IntegerType

derived_fields = [
    StructField("item_popularity", DoubleType()),
    StructField("item_conversion_rate", DoubleType()),
    StructField("item_cart_to_purchase_rate", DoubleType()),
    StructField("price_pct_rank", DoubleType()),
    StructField("price_pct_rank_l2", DoubleType()),
    StructField("item_recency_val", DoubleType()),
    StructField("hour_sin", DoubleType()),
    StructField("hour_cos", DoubleType()),
    StructField("dow_sin", DoubleType()),
    StructField("dow_cos", DoubleType()),
    StructField("item_idx", IntegerType()),
    StructField("category_idx", IntegerType()),
    StructField("category_l2_idx", IntegerType()),
    StructField("category_l3_idx", IntegerType()),
    StructField("brand_idx", IntegerType()),
]
output_schema = StructType(list(train_df.schema.fields) + derived_fields)

def make_transform_fn(vocabs):
    """Closure capturing vocabs for mapInPandas (serialized via cloudpickle)."""
    def _transform(iterator):
        for pdf in iterator:
            yield transform_all_features(pdf, vocabs)
    return _transform

train_sdf = train_df.mapInPandas(make_transform_fn(vocabs), schema=output_schema)
eval_sdf = eval_df.mapInPandas(make_transform_fn(vocabs), schema=output_schema)

train_pdf = train_sdf.toPandas()
eval_pdf = eval_sdf.toPandas()
print(f"\nTrain pandas: {len(train_pdf):,} rows")
print(f"Eval pandas:  {len(eval_pdf):,} rows")

In [0]:
# ---------------------------------------------------------------------------
# Two-tower model adapted for Feature View columns.
# ItemTower: 5 categorical embeddings + 7 derived numeric features
# UserTower: 9 numeric features only (no categorical — simpler, easy to trace)
# ---------------------------------------------------------------------------

class ItemTower(nn.Module):
    def __init__(self, item_vocab_size, category_vocab_size, category_l2_vocab_size,
                 category_l3_vocab_size, brand_vocab_size, embed_dim=EMBED_DIM):
        super().__init__()
        self.item_id_emb = nn.Embedding(item_vocab_size, embed_dim)
        self.category_emb = nn.Embedding(category_vocab_size, embed_dim // 2)
        self.category_l2_emb = nn.Embedding(category_l2_vocab_size, embed_dim // 4)
        self.category_l3_emb = nn.Embedding(category_l3_vocab_size, embed_dim // 4)
        self.brand_emb = nn.Embedding(brand_vocab_size, embed_dim // 2)
        numeric_dim = 7  # log_price, price_pct_rank, price_pct_rank_l2, popularity, conversion, cart_to_purchase, recency
        mlp_in = embed_dim + (embed_dim // 2) * 2 + (embed_dim // 4) * 2 + numeric_dim
        self.mlp = nn.Sequential(
            nn.Linear(mlp_in, 256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, embed_dim),
        )

    def forward(self, item_id, category_id, category_l2_id, category_l3_id, brand_id,
                log_price, price_pct_rank, price_pct_rank_l2, item_popularity,
                item_conversion_rate, item_cart_to_purchase_rate, item_recency):
        x = torch.cat([
            self.item_id_emb(item_id),
            self.category_emb(category_id),
            self.category_l2_emb(category_l2_id),
            self.category_l3_emb(category_l3_id),
            self.brand_emb(brand_id),
            log_price.unsqueeze(-1), price_pct_rank.unsqueeze(-1),
            price_pct_rank_l2.unsqueeze(-1), item_popularity.unsqueeze(-1),
            item_conversion_rate.unsqueeze(-1), item_cart_to_purchase_rate.unsqueeze(-1),
            item_recency.unsqueeze(-1),
        ], dim=-1)
        return TF.normalize(self.mlp(x), p=2, dim=-1)


class UserTower(nn.Module):
    """Simplified user tower: numeric features only (no categorical embedding).
    Matches the Feature View columns available at serving time."""
    def __init__(self, embed_dim=EMBED_DIM):
        super().__init__()
        numeric_dim = 9  # session_event_count, session_cart_value, session_avg_price,
                          # session_price_stddev, session_total_value, hour_sin/cos, dow_sin/cos
        self.mlp = nn.Sequential(
            nn.Linear(numeric_dim, 128), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, embed_dim),
        )

    def forward(self, x):
        return TF.normalize(self.mlp(x), p=2, dim=-1)


class TwoTowerModel(nn.Module):
    def __init__(self, item_tower, user_tower, temperature=0.05):
        super().__init__()
        self.item_tower = item_tower
        self.user_tower = user_tower
        self.temperature = temperature

    def forward(self, user_batch, item_batch):
        user_emb = self.user_tower(**user_batch)
        item_emb = self.item_tower(**item_batch)
        logits = user_emb @ item_emb.T / self.temperature
        labels = torch.arange(logits.size(0), device=logits.device)
        loss = TF.cross_entropy(logits, labels)
        return loss, user_emb, item_emb


# Instantiate on GPU (fp32 per project notes — bf16 + temp=0.05 is unstable)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
item_tower = ItemTower(
    item_vocab_size=len(item_vocab) + 1,
    category_vocab_size=len(category_vocab) + 1,
    category_l2_vocab_size=len(category_l2_vocab) + 1,
    category_l3_vocab_size=len(category_l3_vocab) + 1,
    brand_vocab_size=len(brand_vocab) + 1,
)
user_tower = UserTower()
model = TwoTowerModel(item_tower, user_tower).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Device: {device}")
print(f"Total parameters: {total_params:,}")

In [0]:
# ---------------------------------------------------------------------------
# Dataset, DataLoader, and 1-epoch training loop.
# ---------------------------------------------------------------------------

USER_NUMERIC_COLS = [
    "session_event_count", "session_cart_value", "session_avg_price",
    "session_price_stddev", "session_total_value",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos",
]

class TwoTowerDataset(Dataset):
    def __init__(self, df):
        self.item_id = torch.tensor(df["item_idx"].values, dtype=torch.long)
        self.category_id = torch.tensor(df["category_idx"].values, dtype=torch.long)
        self.category_l2_id = torch.tensor(df["category_l2_idx"].values, dtype=torch.long)
        self.category_l3_id = torch.tensor(df["category_l3_idx"].values, dtype=torch.long)
        self.brand_id = torch.tensor(df["brand_idx"].values, dtype=torch.long)
        self.log_price = torch.tensor(df["item_log_price"].values, dtype=torch.float32)
        self.price_pct_rank = torch.tensor(df["price_pct_rank"].values, dtype=torch.float32)
        self.price_pct_rank_l2 = torch.tensor(df["price_pct_rank_l2"].values, dtype=torch.float32)
        self.item_popularity = torch.tensor(df["item_popularity"].values, dtype=torch.float32)
        self.item_conversion_rate = torch.tensor(df["item_conversion_rate"].values, dtype=torch.float32)
        self.item_cart_to_purchase_rate = torch.tensor(df["item_cart_to_purchase_rate"].values, dtype=torch.float32)
        self.item_recency = torch.tensor(df["item_recency_val"].values, dtype=torch.float32)
        self.user_numeric = torch.tensor(df[USER_NUMERIC_COLS].values, dtype=torch.float32)

    def __len__(self):
        return len(self.item_id)

    def __getitem__(self, idx):
        user_batch = {"x": self.user_numeric[idx]}
        item_batch = {
            "item_id": self.item_id[idx],
            "category_id": self.category_id[idx],
            "category_l2_id": self.category_l2_id[idx],
            "category_l3_id": self.category_l3_id[idx],
            "brand_id": self.brand_id[idx],
            "log_price": self.log_price[idx],
            "price_pct_rank": self.price_pct_rank[idx],
            "price_pct_rank_l2": self.price_pct_rank_l2[idx],
            "item_popularity": self.item_popularity[idx],
            "item_conversion_rate": self.item_conversion_rate[idx],
            "item_cart_to_purchase_rate": self.item_cart_to_purchase_rate[idx],
            "item_recency": self.item_recency[idx],
        }
        return user_batch, item_batch

def collate_fn(batch):
    user_keys = batch[0][0].keys()
    item_keys = batch[0][1].keys()
    user_batch = {k: torch.stack([b[0][k] for b in batch]) for k in user_keys}
    item_batch = {k: torch.stack([b[1][k] for b in batch]) for k in item_keys}
    return user_batch, item_batch

train_loader = DataLoader(TwoTowerDataset(train_pdf), batch_size=BATCH_SIZE,
                          shuffle=True, collate_fn=collate_fn, drop_last=True)
eval_loader = DataLoader(TwoTowerDataset(eval_pdf), batch_size=BATCH_SIZE,
                         shuffle=False, collate_fn=collate_fn, drop_last=True)
print(f"Train batches: {len(train_loader)} | Eval batches: {len(eval_loader)}")

# --- Training loop ---
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
train_history = []

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_loss = 0
    for i, (user_batch, item_batch) in enumerate(train_loader):
        user_batch = {k: v.to(device) for k, v in user_batch.items()}
        item_batch = {k: v.to(device) for k, v in item_batch.items()}
        optimizer.zero_grad()
        loss, _, _ = model(user_batch, item_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        if (i + 1) % 50 == 0:
            print(f"  Epoch {epoch}/{NUM_EPOCHS} — Batch {i+1}/{len(train_loader)}: loss={loss.item():.4f}")

    avg_loss = total_loss / max(len(train_loader), 1)
    train_history.append({"epoch": epoch, "train_loss": avg_loss})
    print(f"Epoch {epoch}/{NUM_EPOCHS} avg train loss: {avg_loss:.4f}\n")

print(f"Training complete ({NUM_EPOCHS} epochs)")

In [0]:
# ---------------------------------------------------------------------------
# Trace both towers, log the user retrieval pyfunc to MLflow, register to UC.
#
# Single run — no staging. The pyfunc handles:
#   session features → user embedding → Vector Search → top-K products
#
# Logged via fe.log_model with a training_set so the FE wrapper knows which
# features to look up at serving time (session features from Lakebase).
# resources=[DatabricksVectorSearchIndex] declares VS access for auth passthrough.
# ---------------------------------------------------------------------------
import tempfile, json, importlib.util
from mlflow.models.resources import DatabricksVectorSearchIndex

# 1. Trace both towers as TorchScript
model.eval()
_artifact_dir = tempfile.mkdtemp()

traced_user = torch.jit.trace(model.user_tower, torch.randn(1, 9).to(device))
user_tower_traced_path = f"{_artifact_dir}/user_tower_traced.pt"
traced_user.save(user_tower_traced_path)

traced_item = torch.jit.trace(model.item_tower, (
    torch.tensor([0], dtype=torch.long).to(device),
    torch.tensor([0], dtype=torch.long).to(device),
    torch.tensor([0], dtype=torch.long).to(device),
    torch.tensor([0], dtype=torch.long).to(device),
    torch.tensor([0], dtype=torch.long).to(device),
    torch.tensor([3.5], dtype=torch.float32).to(device),
    torch.tensor([0.5], dtype=torch.float32).to(device),
    torch.tensor([0.5], dtype=torch.float32).to(device),
    torch.tensor([5.0], dtype=torch.float32).to(device),
    torch.tensor([0.1], dtype=torch.float32).to(device),
    torch.tensor([0.05], dtype=torch.float32).to(device),
    torch.tensor([0.5], dtype=torch.float32).to(device),
))
item_tower_traced_path = f"{_artifact_dir}/item_tower_traced.pt"
traced_item.save(item_tower_traced_path)

# 2. Save pyfunc config (read at serving time by UserTowerRetrievalPyfunc)
pyfunc_config = {
    "embed_dim": EMBED_DIM,
    "user_numeric_features": USER_NUMERIC_COLS,
    "vs_endpoint": config["vs_endpoint"],
    "vs_index_name": config["vs_index_name"],
    "catalog": CATALOG,
    "schema": SCHEMA,
}
pyfunc_config_path = f"{_artifact_dir}/pyfunc_config.json"
with open(pyfunc_config_path, "w") as f:
    json.dump(pyfunc_config, f)

# 3. Import pyfunc class — module name MUST match filename for pickle
retrieval_pyfunc_path = os.path.join(_project_root, "user_retrieval_pyfunc.py")
_spec = importlib.util.spec_from_file_location("user_retrieval_pyfunc", retrieval_pyfunc_path)
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
UserTowerRetrievalPyfunc = _mod.UserTowerRetrievalPyfunc

UC_MODEL_NAME = f"{CATALOG}.{SCHEMA}.two_tower_user_retrieval"

# 4. Single MLflow run: metrics + production pyfunc + UC registration
mlflow.end_run()
with mlflow.start_run(run_name="two_tower_train_deploy") as run:
    run_id = run.info.run_id

    mlflow.log_params({
        "catalog": CATALOG, "schema": SCHEMA,
        "training_start_day": config["training_start_day"],
        "training_end_day": config["training_end_day"],
        "num_epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE,
        "embed_dim": EMBED_DIM,
        "train_rows": len(train_pdf), "eval_rows": len(eval_pdf),
    })
    for entry in train_history:
        mlflow.log_metric("train_loss", entry["train_loss"], step=entry["epoch"])

    # Log the user retrieval pyfunc via fe.log_model (Feature View lineage).
    # The FE wrapper handles session feature lookup from Lakebase at serving time;
    # the pyfunc itself handles user embedding + Vector Search retrieval.
    fe.log_model(
        model=UserTowerRetrievalPyfunc(),
        artifact_path="user_tower_retrieval",
        flavor=mlflow.pyfunc,
        training_set=session_training_set,
        registered_model_name=UC_MODEL_NAME,
        artifacts={
            "user_tower": user_tower_traced_path,
            "config": pyfunc_config_path,
        },
        code_paths=[transforms_module_path, retrieval_pyfunc_path],
        pip_requirements=[
            "--extra-index-url https://download.pytorch.org/whl/cpu",
            "torch", "numpy", "pandas", "databricks-vectorsearch",
        ],
        resources=[DatabricksVectorSearchIndex(index_name=VS_INDEX)],
    )

    # Also log the traced item tower as an artifact (used for embedding generation)
    mlflow.log_artifact(item_tower_traced_path, artifact_path="item_tower")

    print(f"Run: {run_id}")
    print(f"Registered: {UC_MODEL_NAME}")

In [0]:
# ---------------------------------------------------------------------------
# Generate item embeddings from the trained item tower.
# One embedding per unique product in the training data.
# Write to Delta for Vector Search sync.
# ---------------------------------------------------------------------------
model.eval()

# Deduplicate: one row per product with its features
item_features = train_pdf.drop_duplicates("item_idx")[[
    "product_id", "item_idx", "category_idx", "category_l2_idx", "category_l3_idx",
    "brand_idx", "item_log_price", "price_pct_rank", "price_pct_rank_l2",
    "item_popularity", "item_conversion_rate", "item_cart_to_purchase_rate", "item_recency_val",
    "item_category_l1", "item_category_l2", "item_category_l3", "item_brand", "item_price",
]].sort_values("item_idx").reset_index(drop=True)

# Generate embeddings in batches
all_embeddings = []
bs = 1024
with torch.no_grad():
    for start in range(0, len(item_features), bs):
        chunk = item_features.iloc[start:start + bs]
        emb = model.item_tower(
            torch.tensor(chunk["item_idx"].values, dtype=torch.long).to(device),
            torch.tensor(chunk["category_idx"].values, dtype=torch.long).to(device),
            torch.tensor(chunk["category_l2_idx"].values, dtype=torch.long).to(device),
            torch.tensor(chunk["category_l3_idx"].values, dtype=torch.long).to(device),
            torch.tensor(chunk["brand_idx"].values, dtype=torch.long).to(device),
            torch.tensor(chunk["item_log_price"].values, dtype=torch.float32).to(device),
            torch.tensor(chunk["price_pct_rank"].values, dtype=torch.float32).to(device),
            torch.tensor(chunk["price_pct_rank_l2"].values, dtype=torch.float32).to(device),
            torch.tensor(chunk["item_popularity"].values, dtype=torch.float32).to(device),
            torch.tensor(chunk["item_conversion_rate"].values, dtype=torch.float32).to(device),
            torch.tensor(chunk["item_cart_to_purchase_rate"].values, dtype=torch.float32).to(device),
            torch.tensor(chunk["item_recency_val"].values, dtype=torch.float32).to(device),
        )
        all_embeddings.append(emb.cpu().numpy())

embeddings_np = np.vstack(all_embeddings)

# Build DataFrame with product_id, metadata, and embedding vector
emb_df = item_features[["product_id", "item_category_l1", "item_category_l2",
                         "item_category_l3", "item_brand", "item_price"]].copy()
emb_df["embedding"] = [emb.tolist() for emb in embeddings_np]

# Write to Delta
spark_emb_df = spark.createDataFrame(emb_df)
spark_emb_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(EMBEDDING_TABLE)
spark.sql(f"ALTER TABLE {EMBEDDING_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

print(f"Saved {len(emb_df):,} item embeddings to {EMBEDDING_TABLE}")
print(f"Embedding dim: {EMBED_DIM}")
print(f"L2 norm check: {np.linalg.norm(embeddings_np[0]):.4f}")
display(spark.table(EMBEDDING_TABLE).limit(3))

In [0]:
# ---------------------------------------------------------------------------
# Create or sync the Vector Search index from the embeddings Delta table.
# Uses the provisioned endpoint: vs_endpoint
# ---------------------------------------------------------------------------
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

# Check if index exists; create if not
try:
    idx_info = vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=VS_INDEX)
    print(f"Index {VS_INDEX} exists. Syncing...")
    idx_info.sync()
except Exception as e:
    if "not found" in str(e).lower() or "does not exist" in str(e).lower() or "RESOURCE_DOES_NOT_EXIST" in str(e):
        print(f"Creating index {VS_INDEX}...")
        vsc.create_delta_sync_index(
            endpoint_name=VS_ENDPOINT,
            index_name=VS_INDEX,
            source_table_name=EMBEDDING_TABLE,
            primary_key="product_id",
            embedding_vector_column="embedding",
            embedding_dimension=EMBED_DIM,
            pipeline_type="TRIGGERED",
        )
        print(f"Index {VS_INDEX} created on endpoint {VS_ENDPOINT}")
    else:
        raise e

print("\nVector Search index sync initiated.")
print(f"  Endpoint: {VS_ENDPOINT}")
print(f"  Index: {VS_INDEX}")
print(f"  Source: {EMBEDDING_TABLE}")

In [0]:
# ---------------------------------------------------------------------------
# Deploy the user retrieval pyfunc as a model serving endpoint.
# Auto auth passthrough injects VS credentials via resources declaration.
# ---------------------------------------------------------------------------
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput, ServedEntityInput,
)

w = WorkspaceClient()
ENDPOINT_NAME = "two-tower-recommender"

# Get latest model version
from databricks.sdk.service.catalog import ModelVersionInfo
versions = list(w.model_versions.list(full_name=UC_MODEL_NAME))
latest_version = max(v.version for v in versions)
print(f"Deploying {UC_MODEL_NAME} version {latest_version}")

# Delete existing endpoint if present
try:
    w.serving_endpoints.delete(name=ENDPOINT_NAME)
    print(f"Deleted existing endpoint: {ENDPOINT_NAME}")
    import time; time.sleep(5)
except Exception:
    pass

w.serving_endpoints.create(
    name=ENDPOINT_NAME,
    config=EndpointCoreConfigInput(
        name=ENDPOINT_NAME,
        served_entities=[
            ServedEntityInput(
                entity_name=UC_MODEL_NAME,
                entity_version=str(latest_version),
                workload_size="Small",
                scale_to_zero_enabled=True,
                # Automatic auth passthrough: no manual SP credentials needed.
                # resources=[DatabricksVectorSearchIndex(...)] at log time
                # triggers Databricks to provision + rotate credentials.
            )
        ],
    ),
)
print(f"\nEndpoint '{ENDPOINT_NAME}' created. Waiting for deployment...")

# Poll for readiness (use enum comparison, not string matching)
import time
from databricks.sdk.service.serving import EndpointStateReady, EndpointStateConfigUpdate

for i in range(60):
    ep = w.serving_endpoints.get(name=ENDPOINT_NAME)
    ready = ep.state.ready
    config_state = ep.state.config_update
    print(f"  [{i*30}s] ready={ready}, config={config_state}")
    if ready == EndpointStateReady.READY and config_state == EndpointStateConfigUpdate.NOT_UPDATING:
        print("\n\u2713 Endpoint is READY")
        break
    if config_state in (EndpointStateConfigUpdate.UPDATE_FAILED, EndpointStateConfigUpdate.UPDATE_CANCELED):
        print("\n\u2717 Deployment FAILED")
        if ep.pending_config and ep.pending_config.served_entities:
            se = ep.pending_config.served_entities[0]
            print(f"  Error: {se.state.deployment_state_message}")
        break
    time.sleep(30)

In [0]:
# ---------------------------------------------------------------------------
# Test the deployed endpoint with entity keys + request features.
# The endpoint should:
#   1. Look up session features from Lakebase (via feature lineage)
#   2. Look up product features from Lakebase
#   3. Pass all features to the user tower pyfunc
#   4. pyfunc generates embedding → queries Vector Search → returns top-K
# ---------------------------------------------------------------------------

# Get test entity keys from clickstream + product (no Lakebase SQL access needed)
test_row = spark.sql(f"""
    SELECT c.product_id, p.category_l1, p.category_l2, c.user_session
    FROM {TABLE} c
    JOIN {PRODUCT_TABLE} p ON c.product_id = p.product_id
    WHERE p.category_l1 IS NOT NULL
      AND p.category_l2 IS NOT NULL
      AND c.event_time BETWEEN '2019-11-01' AND '2019-11-02'
    LIMIT 1
""").collect()[0]

test_payload = {
    "dataframe_records": [{
        # Entity keys (for Lakebase feature lookup)
        "product_id": test_row.product_id,
        "category_l1": test_row.category_l1,
        "category_l2": test_row.category_l2,
        "user_session": test_row.user_session,
        # RequestSource features (passed directly)
        "relevance": 1,
        "session_cart_value": 0.0,
        "hour_of_day": 14,
        "day_of_week": 3,
    }]
}

print(f"Test payload:")
for k, v in test_payload["dataframe_records"][0].items():
    print(f"  {k}: {v}")

response = w.serving_endpoints.query(
    name=ENDPOINT_NAME,
    **test_payload,
)

print(f"\nResponse:")
print(response.predictions)